In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.neural_network import MLPRegressor


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score


In [ ]:
hourly = pd.read_csv('../01_Data/processed/model_ready_hourly.csv', parse_dates=['date_time'])


In [ ]:
features = ["hour","dayofweek","month","is_weekend","lag_1","lag_24","lag_168","rolling_24","rolling_168"]


## ANN

In [ ]:
cut = int(len(hourly) * 0.8)


In [ ]:
train = hourly.iloc[:cut]
test = hourly.iloc[cut:]


In [ ]:
ann = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32,16), max_iter=200, early_stopping=True, random_state=42))


In [ ]:
ann.fit(train[features], train["energy_kwh"])


In [ ]:
pred_ann = ann.predict(test[features])


In [ ]:
print("ANN MAE", round(mean_absolute_error(test["energy_kwh"], pred_ann), 3))


In [ ]:
print("ANN R2", round(r2_score(test["energy_kwh"], pred_ann), 3))


## LSTM

In [ ]:
hourly = pd.read_csv("../01_Data/processed/hourly_energy.csv", parse_dates=["date_time"])


In [ ]:
y = hourly["energy_kwh"].values.astype("float32")


In [ ]:
train_end = int(len(y) * 0.8)


In [ ]:
mean = y[:train_end].mean()
std = y[:train_end].std()


In [ ]:
y = (y - mean) / std


In [ ]:
seq = 24


In [ ]:
X = []
y_seq = []


In [ ]:
for i in range(seq, len(y)):
    X.append(y[i-seq:i])
    y_seq.append(y[i])


In [ ]:
X = np.array(X, dtype="float32")


In [ ]:
y_seq = np.array(y_seq, dtype="float32")


In [ ]:
pos = np.arange(seq, len(y))


In [ ]:
X_train = X[pos < train_end]
y_train = y_seq[pos < train_end]


In [ ]:
X_test = X[pos >= train_end]
y_test = y_seq[pos >= train_end]


In [ ]:
print(X_train.shape)
print(X_test.shape)


In [ ]:
import torch
from torch import nn


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])


In [ ]:
model = LSTMModel()


In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


In [ ]:
X_train = torch.tensor(X_train).unsqueeze(-1)
y_train = torch.tensor(y_train).unsqueeze(-1)


In [ ]:
X_test = torch.tensor(X_test).unsqueeze(-1)


In [ ]:
for epoch in range(30):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()


In [ ]:
print("Final loss", round(float(loss.item()), 4))


In [ ]:
with torch.no_grad():
    pred_lstm = model(X_test).squeeze().numpy()


In [ ]:
pred_lstm = pred_lstm * std + mean


In [ ]:
y_true = y_test * std + mean


In [ ]:
print("LSTM MAE", round(mean_absolute_error(y_true, pred_lstm), 3))


In [ ]:
print("LSTM R2", round(r2_score(y_true, pred_lstm), 3))


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y_true[-168:], label="Actual")
plt.plot(pred_lstm[-168:], label="LSTM")
plt.legend()
plt.title("LSTM Benchmark")
plt.ylabel("kWh")
plt.show()


In [ ]:
plt.savefig('../04_Visuals/forecasting/10_lstm_benchmark.png', dpi=150)
plt.show()


CNN is not used here because this dataset has time and numeric columns, not images.